# HCMT — Notebook 01: TCGA-BRCA Data Exploration

This notebook walks through:
1. Loading and inspecting the clinical CSV
2. PAM50 class distribution
3. Inspecting WSI feature tensors
4. Genomics data statistics
5. Missing modality analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
%matplotlib inline

DATA_ROOT = Path('../data/tcga_brca')

## 1. Clinical Data

In [ ]:
clinical = pd.read_csv(DATA_ROOT / 'clinical.csv', index_col='patient_id')
print(f'Patients: {len(clinical)}')
print(f'Columns: {list(clinical.columns)}')
clinical.head()

## 2. PAM50 Class Distribution

In [ ]:
PAM50_COLORS = {
    'LumA':   '#6366f1',
    'LumB':   '#10b981',
    'Her2':   '#f59e0b',
    'Basal':  '#ef4444',
    'Normal': '#94a3b8',
}

counts = clinical['PAM50'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = [PAM50_COLORS.get(c, 'gray') for c in counts.index]
axes[0].bar(counts.index, counts.values, color=colors, alpha=0.85, edgecolor='white')
for i, (cls, cnt) in enumerate(counts.items()):
    axes[0].text(i, cnt + 5, str(cnt), ha='center', fontweight='bold')
axes[0].set_title('PAM50 Subtype Counts', fontsize=14, fontweight='bold')
axes[0].set_xlabel('PAM50 Subtype')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('PAM50 Subtype Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nClass imbalance ratio (max/min):', counts.max() / counts.min())

## 3. Clinical Feature Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Age distribution by subtype
for cls, grp in clinical.groupby('PAM50'):
    if 'age_at_diagnosis' in grp.columns:
        axes[0,0].hist(grp['age_at_diagnosis'].dropna(), bins=20,
                       alpha=0.5, label=cls, color=PAM50_COLORS.get(cls))
axes[0,0].set_title('Age at Diagnosis by Subtype')
axes[0,0].set_xlabel('Age')
axes[0,0].legend()

# ER/PR/HER2 status by subtype
for i, marker in enumerate(['er_status', 'pr_status', 'her2_status']):
    if marker in clinical.columns:
        pivot = clinical.groupby(['PAM50', marker]).size().unstack(fill_value=0)
        pivot.plot(kind='bar', ax=axes[0, i+1 if i < 2 else 0],
                   color=['#e2e8f0', '#3b82f6'], alpha=0.8)
        axes[0, i+1 if i < 2 else 0].set_title(f'{marker.replace("_"," ").title()}')
        axes[0, i+1 if i < 2 else 0].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. WSI Feature Inspection

In [ ]:
wsi_dir = DATA_ROOT / 'wsi_features'
wsi_files = list(wsi_dir.glob('*.pt'))
print(f'WSI feature files: {len(wsi_files)}')

# Inspect first file
if wsi_files:
    sample = torch.load(wsi_files[0])
    print(f'Shape: {sample.shape}  — (N_patches, feat_dim)')
    print(f'dtype: {sample.dtype}')
    print(f'min/max/mean: {sample.min():.3f} / {sample.max():.3f} / {sample.mean():.3f}')

    # Patch count distribution
    patch_counts = [torch.load(f).shape[0] for f in wsi_files[:50]]
    plt.figure(figsize=(8, 4))
    plt.hist(patch_counts, bins=20, color='#6366f1', alpha=0.8, edgecolor='white')
    plt.xlabel('Number of Patches per Slide')
    plt.ylabel('Count')
    plt.title('WSI Patch Count Distribution (first 50 patients)')
    plt.axvline(np.median(patch_counts), color='red', linestyle='--',
                label=f'Median: {np.median(patch_counts):.0f}')
    plt.legend()
    plt.show()

## 5. Genomics Feature Inspection

In [ ]:
gen_dir = DATA_ROOT / 'genomics'
gen_files = list(gen_dir.glob('*.pt'))
print(f'Genomics files: {len(gen_files)}')

if gen_files:
    sample = torch.load(gen_files[0])
    print(f'Shape: {sample.shape}  — (n_genes,)')
    print(f'min/max/mean/std: {sample.min():.3f} / {sample.max():.3f} / '
          f'{sample.mean():.3f} / {sample.std():.3f}')

    # Gene expression distribution
    plt.figure(figsize=(10, 4))
    plt.hist(sample.numpy(), bins=100, color='#10b981', alpha=0.8, edgecolor='none')
    plt.xlabel('Normalized Gene Expression')
    plt.ylabel('Gene Count')
    plt.title('Gene Expression Distribution (1 patient)')
    plt.show()

## 6. Missing Modality Analysis

In [ ]:
all_ids = clinical.index.tolist()

availability = {'wsi': [], 'genomics': [], 'radiology': [], 'clinical': []}

for pid in all_ids:
    availability['wsi'].append((DATA_ROOT / 'wsi_features' / f'{pid}.pt').exists())
    availability['genomics'].append((DATA_ROOT / 'genomics' / f'{pid}.pt').exists())
    availability['radiology'].append((DATA_ROOT / 'radiology' / f'{pid}.pt').exists())
    availability['clinical'].append(pid in clinical.index)

avail_df = pd.DataFrame(availability, index=all_ids)

print('Modality availability:')
print(avail_df.mean().to_string())

print(f'\nAll 4 modalities: {avail_df.all(axis=1).sum()} patients ({avail_df.all(axis=1).mean():.1%})')

# Heatmap of availability
plt.figure(figsize=(10, 4))
sample_avail = avail_df.sample(min(100, len(avail_df))).astype(int)
sns.heatmap(sample_avail.T, cmap='Blues', cbar=False,
            xticklabels=False, yticklabels=True)
plt.title('Modality Availability (sample of 100 patients)')
plt.xlabel('Patients')
plt.tight_layout()
plt.show()